# Transcript-Prompted Whisper for Sw->En ST — Kaggle 2x T4 Edition

End-to-end pipeline tuned for Kaggle's 2x T4 environment (vs the EC2 4x L4 version in `05_whisper_st_train_eval.ipynb`).

**Differences from the EC2 version:**
- `fp16` instead of `bf16` (T4 has no native bf16)
- 2-GPU DDP via `accelerate launch --num_processes 2`
- Skips Whisper-medium variants (12-hr session limit)
- Trains 2 variants (small baseline, small hint-augmented) instead of 4

**Total runtime: ~8-10 hrs in one session.** Skip-if-done guards on every step let you re-launch and resume.

**Step order:**
1. Install deps + setup
2. Configure accelerate for 2-GPU DDP
3. Build bidirectional lexicon (~30 sec)
4. Build FLEURS sw_ke test set (~5 min)
5. Sanity-check KenSpeech indexing (~30 sec)
6. Restore sw->en translations locally (~5 min)
7. Train Whisper-small baseline (~3-4 hrs)
8. Train Whisper-small hint-augmented (~3-4 hrs)
9. Generate predictions for 5 variants (~1.5 hrs)
10. BLEU + chrF evaluation (~1 min)
11. Persist results to Kaggle dataset version

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy sentencepiece sacrebleu "datasets<4.0"

In [ ]:
# HuggingFace authentication (for FLEURS test download)
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HF login OK')
except Exception as e:
    print(f'HF token not set ({e}). FLEURS download may rate-limit.')

In [ ]:
# Paths and config
import os, sys, json, torch
from pathlib import Path

# Code dataset (where the whisper_st module lives)
REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

# Input data (Kaggle datasets)
KENSPEECH_DIR = '/kaggle/input/kenspeech-sw'
SW2EN_DIR = '/kaggle/input/datasets/victormugambi/mambosw2en'
EN2SW_DIR = '/kaggle/input/datasets/victormugambi/mamboen2sw'

# Working directories
WORK_DIR = '/kaggle/working'
RUN_ROOT = f'{WORK_DIR}/runs'
LEXICON_PATH = f'{RUN_ROOT}/lexicon.jsonl'
FLEURS_TEST_DIR = f'{RUN_ROOT}/fleurs_sw_ke_test'
TRANSLATION_DIR = f'{WORK_DIR}/hibiki-sw/translations/sw2en'
PRED_DIR = f'{RUN_ROOT}/predictions'
MODEL_BASELINE = f'{RUN_ROOT}/whisper_small_baseline'
MODEL_HINT = f'{RUN_ROOT}/whisper_small_hint'

for d in [RUN_ROOT, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

# GPU check
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Path verification
print('\nPath check:')
for label, path in [
    ('Repo',           REPO_DIR),
    ('KenSpeech',      KENSPEECH_DIR),
    ('sw2en dataset',  SW2EN_DIR),
    ('en2sw dataset',  EN2SW_DIR),
]:
    print(f'  {label:18s}: {"OK" if os.path.exists(path) else "MISSING"}  {path}')

## Step 2: Configure Accelerate for 2-GPU DDP

Kaggle doesn't persist `~/.cache/huggingface/accelerate/` between sessions, so we write the config every time.

In [ ]:
import os
accelerate_cfg_dir = os.path.expanduser('~/.cache/huggingface/accelerate')
os.makedirs(accelerate_cfg_dir, exist_ok=True)
with open(f'{accelerate_cfg_dir}/default_config.yaml', 'w') as f:
    f.write('''compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
mixed_precision: fp16
num_processes: 2
gpu_ids: 0,1
machine_rank: 0
num_machines: 1
use_cpu: false
''')
print('Accelerate configured for 2x T4 fp16')
!cat ~/.cache/huggingface/accelerate/default_config.yaml

## Step 3: Build Bidirectional Lexicon

Uses both sw->en and en->sw alignments. Bidirectional pairs are flagged as high-confidence.

In [ ]:
# Verify alignment subdirectories exist
for label, path in [('sw2en', f'{SW2EN_DIR}/alignments/sw2en'),
                    ('en2sw', f'{EN2SW_DIR}/alignments/en2sw')]:
    if os.path.isdir(path):
        n = len([f for f in os.listdir(path) if f.endswith('.json') and f != 'index.jsonl'])
        print(f'  [{label}] {n} alignment files at {path}')
    else:
        print(f'  [{label}] MISSING at {path}  -- listing dataset root:')
        ds_root = path.split('/alignments/')[0]
        if os.path.isdir(ds_root):
            for entry in sorted(os.listdir(ds_root))[:25]:
                print(f'      {entry}')

In [ ]:
# Build the lexicon (skip if already done)
if os.path.exists(LEXICON_PATH) and os.path.getsize(LEXICON_PATH) > 0:
    n = sum(1 for _ in open(LEXICON_PATH))
    print(f'Lexicon exists with {n} entries -- delete to rebuild')
else:
    !cd {REPO_DIR} && python whisper_st/build_lexicon.py \
        --alignments sw2en={SW2EN_DIR}/alignments/sw2en \
                     en2sw={EN2SW_DIR}/alignments/en2sw \
        --output_path {LEXICON_PATH} \
        --min_freq 2 \
        --min_alignment_consistency 0.4 \
        --include_wikidata

In [ ]:
# Inspect lexicon
n_total = 0
n_bidir = 0
examples_bidir = []
examples_uni = []
with open(LEXICON_PATH) as f:
    for line in f:
        e = json.loads(line)
        n_total += 1
        if e.get('bidirectional'):
            n_bidir += 1
            if len(examples_bidir) < 10:
                examples_bidir.append(e)
        elif len(examples_uni) < 5:
            examples_uni.append(e)
print(f'Total entries: {n_total} | Bidirectional (high-confidence): {n_bidir}')
print('\nTop 10 bidirectional:')
for e in examples_bidir:
    print(f'  {e["sw"]:25s} -> {e["en"]:30s} (freq={e["freq"]}, consistency={e.get("consistency")})')
print('\nTop 5 single-direction:')
for e in examples_uni:
    print(f'  {e["sw"]:25s} -> {e["en"]:30s} (freq={e["freq"]}, dirs={e.get("directions")})')

## Step 4: Build FLEURS sw_ke Test Reference Set

FLEURS is parallel by `id`, so we use sw_ke audio with paired en_us references.

In [ ]:
if os.path.exists(f'{FLEURS_TEST_DIR}/refs.jsonl') and os.path.getsize(f'{FLEURS_TEST_DIR}/refs.jsonl') > 0:
    n = sum(1 for _ in open(f'{FLEURS_TEST_DIR}/refs.jsonl'))
    n_wavs = len(list(Path(f'{FLEURS_TEST_DIR}/audio').glob('*.wav'))) if os.path.exists(f'{FLEURS_TEST_DIR}/audio') else 0
    print(f'FLEURS test exists: {n} refs, {n_wavs} wavs -- skipping')
else:
    from datasets import load_dataset
    import soundfile as sf, numpy as np
    os.makedirs(f'{FLEURS_TEST_DIR}/audio', exist_ok=True)

    print('Downloading FLEURS sw_ke test...')
    sw_ds = load_dataset('google/fleurs', 'sw_ke', split='test', trust_remote_code=True)
    print(f'  {len(sw_ds)} samples')

    print('Downloading FLEURS en_us test (for parallel English references)...')
    en_ds = load_dataset('google/fleurs', 'en_us', split='test', trust_remote_code=True)
    en_by_id = {s['id']: s['transcription'] for s in en_ds}
    print(f'  {len(en_by_id)} samples')

    n_paired = 0
    with open(f'{FLEURS_TEST_DIR}/refs.jsonl', 'w', encoding='utf-8') as f:
        for i, sample in enumerate(sw_ds):
            sid = sample['id']
            if sid not in en_by_id:
                continue
            wav_name = f'fleurs_sw_ke_{i:05d}.wav'
            audio = sample['audio']
            sf.write(f'{FLEURS_TEST_DIR}/audio/{wav_name}',
                     np.asarray(audio['array'], dtype=np.float32),
                     audio['sampling_rate'])
            f.write(json.dumps({
                'audio': wav_name,
                'reference_en': en_by_id[sid],
                'reference_sw': sample['transcription'],
                'id': sid,
            }, ensure_ascii=False) + '\n')
            n_paired += 1
    print(f'\nWrote {n_paired} parallel test pairs to {FLEURS_TEST_DIR}/')

## Step 5: Sanity-Check KenSpeech Indexing

Confirms `sample_idx` in translation JSONs matches what `KenSpeechLoader[idx]` returns. **STOP and patch the dataset class if you see MISMATCH.**

In [ ]:
import random
from data.prepare.kenspeech_loader import KenSpeechLoader
ks = KenSpeechLoader(load_audio=False, local_dir=KENSPEECH_DIR)

# Find translations subdirectory
trans_root = None
for cand in [f'{SW2EN_DIR}/translations/sw2en', f'{SW2EN_DIR}/translations']:
    if os.path.isdir(cand):
        trans_root = cand
        break
if trans_root is None:
    raise RuntimeError(f'Could not find translations under {SW2EN_DIR}. Inspect contents:\n' +
                       '\n'.join(sorted(os.listdir(SW2EN_DIR))))

files = sorted(Path(trans_root).glob('*.json'))
print(f'Translations root: {trans_root} ({len(files)} files)')

random.seed(0)
n_match, n_mismatch = 0, 0
for jp in random.sample(files, min(5, len(files))):
    data = json.load(open(jp))
    si = data['sample_idx']
    sw_t = data['source_text'].strip()
    sw_k = ks[si]['sentence'].strip()
    match = sw_t == sw_k
    print(f'  sample_idx={si}: {"MATCH" if match else "MISMATCH"}')
    if not match:
        print(f'    translation src: {sw_t[:80]}')
        print(f'    kenspeech[{si}]: {sw_k[:80]}')
    n_match += int(match); n_mismatch += int(not match)

print(f'\nResult: {n_match} match, {n_mismatch} mismatch')
if n_mismatch > 0:
    raise RuntimeError('STOP: KenSpeech sample_idx does not match loader order. Dataset class needs a remapping patch before training.')

## Step 6: Restore Translations Locally

Trainer reads from local working dir; copy from the saved Kaggle dataset.

In [ ]:
import shutil
if not os.path.isdir(TRANSLATION_DIR) or not os.listdir(TRANSLATION_DIR):
    os.makedirs(TRANSLATION_DIR, exist_ok=True)
    shutil.copytree(trans_root, TRANSLATION_DIR, dirs_exist_ok=True)
n = len(list(Path(TRANSLATION_DIR).glob('*.json')))
print(f'Translations ready: {n} files at {TRANSLATION_DIR}')

## Step 7: Train Whisper-small Baseline (no lexicon hints)

**Expected runtime: ~3-4 hrs on 2x T4.** First training run; this is the main risk point.

**Watch the first ~50 steps for:**
- `loss: nan` -> drop `--lr` to `5e-6` and restart
- OOM -> reduce `--batch_size` to 4 with `--grad_accum 4`
- DDP errors -> set `LAUNCH=python` to fall back to single-GPU

In [ ]:
if os.path.exists(f'{MODEL_BASELINE}/final/config.json'):
    print(f'Baseline model already trained at {MODEL_BASELINE}/final -- skipping')
else:
    !cd {REPO_DIR} && accelerate launch --num_processes 2 --mixed_precision fp16 \
        whisper_st/train.py \
        --base_model openai/whisper-small \
        --translations_dir {TRANSLATION_DIR} \
        --kenspeech_dir {KENSPEECH_DIR} \
        --output_dir {MODEL_BASELINE} \
        --batch_size 8 --grad_accum 1 \
        --lr 1e-5 --epochs 3 --warmup_steps 100 \
        --ctc_loss_weight 0.3 \
        --precision fp16 \
        --num_workers 2

### CHECKPOINT — save baseline model before continuing

If you're worried about session timeout, click **Save Version** in the Kaggle UI now to persist the baseline model. The `if exists, skip` guards above mean re-running the notebook will skip already-trained models.

## Step 8: Train Whisper-small with Hint-Augmented Prompts

Same architecture, but during training the dataset injects lexicon hints into the source-transcript portion of the decoder prompt (with `hint_prob=0.5`). Teaches the model to use lexicon hints at inference.

**Expected runtime: ~3-4 hrs on 2x T4.**

In [ ]:
if os.path.exists(f'{MODEL_HINT}/final/config.json'):
    print(f'Hint model already trained at {MODEL_HINT}/final -- skipping')
else:
    !cd {REPO_DIR} && accelerate launch --num_processes 2 --mixed_precision fp16 \
        whisper_st/train.py \
        --base_model openai/whisper-small \
        --translations_dir {TRANSLATION_DIR} \
        --kenspeech_dir {KENSPEECH_DIR} \
        --output_dir {MODEL_HINT} \
        --batch_size 8 --grad_accum 1 \
        --lr 1e-5 --epochs 3 --warmup_steps 100 \
        --ctc_loss_weight 0.3 \
        --precision fp16 \
        --num_workers 2 \
        --lexicon_path {LEXICON_PATH} \
        --hint_prob 0.5

## Step 9: Generate Predictions for 5 Variants

1. **vanilla_small** — Whisper-small zero-shot Sw->En (no fine-tuning, no lexicon)
2. **baseline_nolex** — fine-tuned baseline, no lexicon at inference
3. **baseline_lex** — fine-tuned baseline, lexicon at inference (model never trained on hint format — should be neutral or slight regression)
4. **hint_nolex** — hint-trained model, no lexicon at inference
5. **hint_lex** — hint-trained model + lexicon at inference (the headline system)

Each takes ~15-30 min for ~250 FLEURS test wavs on a single T4.

In [ ]:
PRED_VANILLA = f'{PRED_DIR}/preds_vanilla_small.jsonl'
if os.path.exists(PRED_VANILLA) and os.path.getsize(PRED_VANILLA) > 0:
    print(f'Vanilla predictions exist -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/baseline_vanilla.py \
        --base_model openai/whisper-small \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_VANILLA}

In [ ]:
PRED_BASELINE_NOLEX = f'{PRED_DIR}/preds_baseline_nolex.jsonl'
if os.path.exists(PRED_BASELINE_NOLEX) and os.path.getsize(PRED_BASELINE_NOLEX) > 0:
    print(f'baseline_nolex predictions exist -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_BASELINE}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_BASELINE_NOLEX}

In [ ]:
PRED_BASELINE_LEX = f'{PRED_DIR}/preds_baseline_lex.jsonl'
if os.path.exists(PRED_BASELINE_LEX) and os.path.getsize(PRED_BASELINE_LEX) > 0:
    print(f'baseline_lex predictions exist -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_BASELINE}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_BASELINE_LEX} \
        --lexicon_path {LEXICON_PATH}

In [ ]:
PRED_HINT_NOLEX = f'{PRED_DIR}/preds_hint_nolex.jsonl'
if os.path.exists(PRED_HINT_NOLEX) and os.path.getsize(PRED_HINT_NOLEX) > 0:
    print(f'hint_nolex predictions exist -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_HINT}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_HINT_NOLEX}

In [ ]:
PRED_HINT_LEX = f'{PRED_DIR}/preds_hint_lex.jsonl'
if os.path.exists(PRED_HINT_LEX) and os.path.getsize(PRED_HINT_LEX) > 0:
    print(f'hint_lex predictions exist -- skipping')
else:
    !cd {REPO_DIR} && python whisper_st/inference.py \
        --model_dir {MODEL_HINT}/final \
        --audio_dir {FLEURS_TEST_DIR}/audio \
        --output_path {PRED_HINT_LEX} \
        --lexicon_path {LEXICON_PATH}

## Step 10: BLEU + chrF Evaluation

All 5 variants scored against the same FLEURS sw_ke -> en_us references. Skips any variant whose predictions are missing.

In [ ]:
variants = []
for name, path in [
    ('vanilla_small',    PRED_VANILLA),
    ('baseline_nolex',   PRED_BASELINE_NOLEX),
    ('baseline_lex',     PRED_BASELINE_LEX),
    ('hint_nolex',       PRED_HINT_NOLEX),
    ('hint_lex',         PRED_HINT_LEX),
]:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        variants.append(f'{name}={path}')
    else:
        print(f'  Missing: {name} ({path})')

preds_args = ' '.join(variants)
print(f'Scoring {len(variants)} variants...\n')
!cd {REPO_DIR} && python whisper_st/eval_bleu.py \
    --references_path {FLEURS_TEST_DIR}/refs.jsonl \
    --predictions {preds_args} \
    --show_examples 12 \
    | tee {RUN_ROOT}/results.txt

## Step 11: Persist Results

Save everything in `/kaggle/working/runs/` so it survives the session ending.

In [ ]:
# Show what's there to be saved
print('=== Output Summary ===')
for name, path in [
    ('Lexicon',            LEXICON_PATH),
    ('FLEURS test refs',   f'{FLEURS_TEST_DIR}/refs.jsonl'),
    ('Baseline model',     f'{MODEL_BASELINE}/final'),
    ('Hint model',         f'{MODEL_HINT}/final'),
    ('Predictions',        PRED_DIR),
    ('Results',            f'{RUN_ROOT}/results.txt'),
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            n = sum(1 for _ in Path(path).rglob('*') if _.is_file())
            size_mb = sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file()) / 1e6
            print(f'  {name:25s}: {n} files, {size_mb:.1f} MB at {path}')
        else:
            size_mb = os.path.getsize(path) / 1e6
            print(f'  {name:25s}: {size_mb:.2f} MB at {path}')
    else:
        print(f'  {name:25s}: [NOT FOUND] {path}')

print('\n=== Persistence ===')
print('Click "Save Version" -> "Save Output Only" in the Kaggle UI to')
print('persist /kaggle/working/runs/ as a versioned dataset.')

In [ ]:
# Optional: zip everything for easier download
import shutil
ZIP_PATH = '/kaggle/working/whisper_st_results'
if not os.path.exists(ZIP_PATH + '.zip'):
    shutil.make_archive(ZIP_PATH, 'zip', root_dir=RUN_ROOT)
size_mb = os.path.getsize(ZIP_PATH + '.zip') / 1e6
print(f'Zipped runs/ -> {ZIP_PATH}.zip ({size_mb:.1f} MB)')
print('Download from the Kaggle right-side Output panel.')

## Recovery / Resume Guide

If the session ends mid-run:

1. **Click "Save Version"** in the UI to persist whatever's in `/kaggle/working/runs/`
2. **In a new session**, attach the saved-version dataset as input
3. **Add a cell at the top** that copies the saved state back into `/kaggle/working/runs/`:
   ```python
   import shutil
   prev_run = '/kaggle/input/<your-saved-version-slug>/runs'
   if os.path.isdir(prev_run):
       shutil.copytree(prev_run, RUN_ROOT, dirs_exist_ok=True)
       print(f'Restored from {prev_run}')
   ```
4. **Re-run the whole notebook** — every step has skip-if-done guards, so completed work is reused.

## Time Budget Per Session

If you want to split across sessions instead of one-shot:

| Session | Steps | Time |
|---|---|---|
| 1 | Steps 1-6 (setup + lexicon + test set + restore translations) | ~15 min |
| 2 | Step 7 (baseline training) + Save Version | ~3-4 hrs |
| 3 | Step 8 (hint training) + Save Version | ~3-4 hrs |
| 4 | Steps 9-11 (inference + eval + persist) | ~2 hrs |

Or one-shot if you trust the 12-hr session limit (8-10 hrs total).